# Router Crop Identification

This notebook keeps only the thin router path: bootstrap, optional dependency install, image upload, and VLM crop/part analysis.

The router is prepared once per runtime and then reused for new images, so repeated trials do not redownload or reload the models unless you explicitly clear the session cache.

The returned payload keeps mirrored top-level crop fields and a structured `router` block with router status, primary detection, and detection count. It does not load any crop adapter.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
    Path('/content/drive/MyDrive/bitirme projesi'),
    Path('/content/drive/MyDrive/bitirmeprojesi'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _mount_drive_inline() -> None:
    if not _running_in_colab_inline():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Drive mount skipped during bootstrap: {exc}')

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    _mount_drive_inline()
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repository bootstrap failed. Set AADS_REPO_ROOT to an existing checkout, or provide '
        'GH_TOKEN/GITHUB_TOKEN for private GitHub auto-clone.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    mount_drive_if_available()
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

print(ROOT)


In [ ]:
from scripts.colab_repo_bootstrap import login_and_check_hf_token, running_in_colab
from scripts.colab_router_adapter_inference import clear_router_cache, ensure_router_ready, run_inference

# Notebook-first router controls (no codebase edits needed).
NOTEBOOK_PROFILE = 'a100_colab_default'  # one of: custom, a100_colab_default, cpu_debug

INFERENCE_PROFILES = {
    'custom': {},
    'a100_colab_default': {
        'CONFIG_ENV': 'colab',
        'DEVICE': 'cuda',
        'REQUIRE_HF_LOGIN': False,
        'CROP_HINT': None,
        'PART_HINT': None,
        'FORCE_UPLOAD_IF_NO_IMAGE': True,
    },
    'cpu_debug': {
        'CONFIG_ENV': 'colab',
        'DEVICE': 'cpu',
        'REQUIRE_HF_LOGIN': False,
        'CROP_HINT': None,
        'PART_HINT': None,
        'FORCE_UPLOAD_IF_NO_IMAGE': False,
    },
}

CONFIG_ENV = 'colab'
DEVICE = 'cuda'
REQUIRE_HF_LOGIN = False
CROP_HINT = None
PART_HINT = None
IMAGE_PATH = None  # Optional default image path for the analysis cell.
FORCE_UPLOAD_IF_NO_IMAGE = True
PRELOAD_ROUTER = True
RESET_ROUTER_CACHE = False

profile = dict(INFERENCE_PROFILES.get(NOTEBOOK_PROFILE, {}))
CONFIG_ENV = str(profile.get('CONFIG_ENV', CONFIG_ENV))
DEVICE = str(profile.get('DEVICE', DEVICE))
REQUIRE_HF_LOGIN = bool(profile.get('REQUIRE_HF_LOGIN', REQUIRE_HF_LOGIN))
CROP_HINT = profile.get('CROP_HINT', CROP_HINT)
PART_HINT = profile.get('PART_HINT', PART_HINT)
FORCE_UPLOAD_IF_NO_IMAGE = bool(profile.get('FORCE_UPLOAD_IF_NO_IMAGE', FORCE_UPLOAD_IF_NO_IMAGE))
PRELOAD_ROUTER = bool(profile.get('PRELOAD_ROUTER', PRELOAD_ROUTER))
RESET_ROUTER_CACHE = bool(profile.get('RESET_ROUTER_CACHE', RESET_ROUTER_CACHE))

# Optional final overrides with highest precedence.
INFERENCE_OVERRIDES = {}
if INFERENCE_OVERRIDES:
    CONFIG_ENV = str(INFERENCE_OVERRIDES.get('CONFIG_ENV', CONFIG_ENV))
    DEVICE = str(INFERENCE_OVERRIDES.get('DEVICE', DEVICE))
    REQUIRE_HF_LOGIN = bool(INFERENCE_OVERRIDES.get('REQUIRE_HF_LOGIN', REQUIRE_HF_LOGIN))
    CROP_HINT = INFERENCE_OVERRIDES.get('CROP_HINT', CROP_HINT)
    PART_HINT = INFERENCE_OVERRIDES.get('PART_HINT', PART_HINT)
    IMAGE_PATH = INFERENCE_OVERRIDES.get('IMAGE_PATH', IMAGE_PATH)
    FORCE_UPLOAD_IF_NO_IMAGE = bool(INFERENCE_OVERRIDES.get('FORCE_UPLOAD_IF_NO_IMAGE', FORCE_UPLOAD_IF_NO_IMAGE))
    PRELOAD_ROUTER = bool(INFERENCE_OVERRIDES.get('PRELOAD_ROUTER', PRELOAD_ROUTER))
    RESET_ROUTER_CACHE = bool(INFERENCE_OVERRIDES.get('RESET_ROUTER_CACHE', RESET_ROUTER_CACHE))

print(
    f'[CONFIG] profile={NOTEBOOK_PROFILE} env={CONFIG_ENV} device={DEVICE} '
    f'crop_hint={CROP_HINT} part_hint={PART_HINT} preload_router={PRELOAD_ROUTER}'
)

HF_READY = login_and_check_hf_token(print_fn=print)
if REQUIRE_HF_LOGIN and not HF_READY:
    raise RuntimeError('HF login required by notebook settings, but authentication failed.')
if not HF_READY:
    print('[HF] Continuing without an authenticated session. Gated model downloads may fail.')

if RESET_ROUTER_CACHE:
    clear_router_cache()
    print('[ROUTER] Session cache cleared.')

if CROP_HINT:
    print('[ROUTER] crop_hint set; preload skipped because the router will be bypassed.')
elif PRELOAD_ROUTER:
    ensure_router_ready(
        config_env=CONFIG_ENV,
        device=DEVICE,
        status_printer=print,
    )
    print('[NOTEBOOK] Router is cached for this runtime. Rerun the next cell with a new image only.')
else:
    print('[NOTEBOOK] PRELOAD_ROUTER is False. The first analysis call in the next cell will initialize the router.')


In [ ]:
ANALYSIS_IMAGE_PATH = IMAGE_PATH  # Set to a new file path, or keep None to upload a new image.
UPLOAD_NEW_IMAGE = ANALYSIS_IMAGE_PATH is None and FORCE_UPLOAD_IF_NO_IMAGE

if ANALYSIS_IMAGE_PATH is None and UPLOAD_NEW_IMAGE:
    if not running_in_colab():
        raise ValueError('Set ANALYSIS_IMAGE_PATH manually when running outside Colab.')
    from google.colab import files
    uploaded = files.upload()
    ANALYSIS_IMAGE_PATH = next(iter(uploaded.keys()))

if ANALYSIS_IMAGE_PATH is None:
    raise ValueError('ANALYSIS_IMAGE_PATH is required when upload is disabled.')

# Rerun this cell with a different ANALYSIS_IMAGE_PATH to try another image without reloading the router.
result = run_inference(
    ANALYSIS_IMAGE_PATH,
    config_env=CONFIG_ENV,
    crop_hint=CROP_HINT,
    part_hint=PART_HINT,
    device=DEVICE,
    status_printer=print,
)
result